# 02 — Data Cleaning
**Project:** AI-Powered E-Commerce Customer Churn Analytics & Retention Decision Support System  
**Author:** Sumit Raj  

**Purpose:** Apply the reproducible cleaning pipeline, enforce the qualifying-purchase rule,
calculate Revenue, and save the processed outputs to `data/processed/`.
All counts are shown before and after each step for full traceability.


In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import pandas as pd
from src.data.load import load_raw
from src.data.clean import (
    audit_raw,
    remove_exact_duplicates,
    flag_cancellations,
    filter_qualifying_purchases,
    filter_eligible_customers,
    get_eligible_customer_ids,
    save_qualified_transactions,
    save_audit_json,
)
from src.config import SNAPSHOT_DATE, PROCESSED_DIR

## Step 1 — Load Raw Data & Pre-Cleaning Audit


In [2]:
df_raw = load_raw()
rows_raw = len(df_raw)
print(f'Raw rows loaded: {rows_raw:,}')

raw_report = audit_raw(df_raw)
print('\n=== Pre-cleaning audit ===')
for k, v in raw_report.items():
    print(f'  {k}: {v}')

Raw rows loaded: 541,909



=== Pre-cleaning audit ===
  raw_rows: 541909
  raw_columns: 8
  missing_customer_id: 135080
  missing_description: 1454
  exact_duplicates: 5268
  cancellation_rows: 9288
  negative_quantity_rows: 10624
  zero_quantity_rows: 0
  non_positive_quantity_rows: 10624
  negative_unit_price_rows: 2
  zero_unit_price_rows: 2515
  non_positive_unit_price_rows: 2517
  date_parse_failures: 0
  date_min: 2010-12-01
  date_max: 2011-12-09
  unique_invoices: 25900
  unique_products: 4070
  unique_customers: 4372
  country_count: 38


## Step 2 — Schema & DateTime Validation


In [3]:
print('InvoiceDate dtype:', df_raw['InvoiceDate'].dtype)
print('Date parse failures:', df_raw['InvoiceDate'].isna().sum())
print('Date range:', df_raw['InvoiceDate'].min(), 'to', df_raw['InvoiceDate'].max())

InvoiceDate dtype: datetime64[us]
Date parse failures: 0
Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00


## Step 3 — Normalise Text Fields
Fill missing Description with 'Unknown' for product-level analysis.  
This does NOT affect the qualifying-purchase filter.


In [4]:
df_normalised = df_raw.copy()
missing_desc_before = df_normalised['Description'].isna().sum()
df_normalised['Description'] = df_normalised['Description'].fillna('Unknown')
missing_desc_after = df_normalised['Description'].isna().sum()
print(f'Missing Description before: {missing_desc_before:,}')
print(f'Missing Description after : {missing_desc_after:,}  (filled with "Unknown")')

Missing Description before: 1,454
Missing Description after : 0  (filled with "Unknown")


## Step 4 — Remove Exact Duplicates


In [5]:
rows_before_dedup = len(df_normalised)
df_dedup = remove_exact_duplicates(df_normalised)
rows_after_dedup = len(df_dedup)
duplicates_removed = rows_before_dedup - rows_after_dedup

print(f'rows_before_dedup     : {rows_before_dedup:,}')
print(f'duplicate_rows_removed: {duplicates_removed:,}')
print(f'rows_after_dedup      : {rows_after_dedup:,}')

rows_before_dedup     : 541,909
duplicate_rows_removed: 5,268
rows_after_dedup      : 536,641


## Step 5 — Flag Cancellations


In [6]:
df_flagged = flag_cancellations(df_dedup)
n_cancel = df_flagged['IsCancellation'].sum()
print(f'IsCancellation == True  : {n_cancel:,}')
print(f'IsCancellation == False : {(~df_flagged["IsCancellation"]).sum():,}')

IsCancellation == True  : 9,251
IsCancellation == False : 527,390


## Step 6 — Audit Invalid Quantity and UnitPrice


In [7]:
print('In de-duplicated dataset:')
print(f'  Quantity <= 0  : {(df_flagged["Quantity"] <= 0).sum():,}')
print(f'  UnitPrice <= 0 : {(df_flagged["UnitPrice"] <= 0).sum():,}')

In de-duplicated dataset:
  Quantity <= 0  : 10,587
  UnitPrice <= 0 : 2,512


## Step 7 — Audit Missing CustomerID
Missing CustomerID rows are excluded from the customer-level dataset but counted here.


In [8]:
n_missing_cid = df_flagged['CustomerID'].isna().sum()
print(f'Rows with missing CustomerID (post-dedup): {n_missing_cid:,}')
print(f'  These will be excluded from the qualifying purchase dataset.')

Rows with missing CustomerID (post-dedup): 135,037
  These will be excluded from the qualifying purchase dataset.


## Step 8 — Apply Qualifying-Purchase Filter & Calculate Revenue

The four conditions (all must be True):
1. `CustomerID` is not null  
2. `InvoiceNo` does NOT start with `'C'`  
3. `Quantity > 0`  
4. `UnitPrice > 0`  

Revenue = Quantity × UnitPrice — calculated only on qualifying rows.


In [9]:
df_qualified = filter_qualifying_purchases(df_flagged)
rows_qualified = len(df_qualified)
qualified_customers = df_qualified['CustomerID'].nunique()

print(f'Qualifying purchase rows : {rows_qualified:,}')
print(f'Qualifying customers     : {qualified_customers:,}')
print()
print('Revenue validation:')
print(f'  min Revenue : {df_qualified["Revenue"].min():,.2f}')
print(f'  max Revenue : {df_qualified["Revenue"].max():,.2f}')
print(f'  Revenue < 0 : {(df_qualified["Revenue"] < 0).sum():,}  (must be 0)')
print()
print(df_qualified[['InvoiceNo', 'Quantity', 'UnitPrice', 'Revenue']].describe())

Qualifying purchase rows : 392,692
Qualifying customers     : 4,338

Revenue validation:
  min Revenue : 0.00
  max Revenue : 168,469.60
  Revenue < 0 : 0  (must be 0)



            Quantity      UnitPrice        Revenue
count  392692.000000  392692.000000  392692.000000
mean       13.119702       3.125914      22.631500
std       180.492832      22.241836     311.099224
min         1.000000       0.001000       0.001000
25%         2.000000       1.250000       4.950000
50%         6.000000       1.950000      12.450000
75%        12.000000       3.750000      19.800000
max     80995.000000    8142.750000  168469.600000


## Step 9 — Row Counts at Each Stage


In [10]:
print('=== Row counts at each major stage ===')
print(f'  Raw rows loaded             : {rows_raw:,}')
print(f'  After deduplication         : {rows_after_dedup:,}  (removed {rows_raw - rows_after_dedup:,})')
print(f'  Qualifying purchases        : {rows_qualified:,}')
print(f'  Rows removed by cleaning    : {rows_after_dedup - rows_qualified:,}')

=== Row counts at each major stage ===
  Raw rows loaded             : 541,909
  After deduplication         : 536,641  (removed 5,268)
  Qualifying purchases        : 392,692
  Rows removed by cleaning    : 143,949


## Step 10 — Customer Eligibility Summary (Phase 3 Preview, No RFM/Churn Computed Here)


In [11]:
eligible_ids = get_eligible_customer_ids(df_qualified, snapshot=SNAPSHOT_DATE)
print(f'SNAPSHOT_DATE    : {SNAPSHOT_DATE.date()}')
print(f'Eligible customers (observation period, ≥2 invoices, ≥30-day span): {len(eligible_ids):,}')
print(f'All qualifying customers: {qualified_customers:,}')

SNAPSHOT_DATE    : 2011-08-31
Eligible customers (observation period, ≥2 invoices, ≥30-day span): 1,761
All qualifying customers: 4,338


## Step 11 — Save Processed Outputs


In [12]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save qualifying transactions CSV
csv_path = save_qualified_transactions(df_qualified, output_dir=PROCESSED_DIR)
print(f'Saved: {csv_path}')

# Save audit JSON (use raw_report from Step 1 — counts are from pre-dedup raw data)
audit_path = save_audit_json(raw_report, df_qualified, output_dir=PROCESSED_DIR)
print(f'Saved: {audit_path}')

Saved: D:\IBM_SkillsBuild_Data_Analytics_AI_Internship_2026\ecommerce-churn-analytics\data\processed\qualified_transactions.csv
Saved: D:\IBM_SkillsBuild_Data_Analytics_AI_Internship_2026\ecommerce-churn-analytics\data\processed\data_quality_audit.json


## Step 12 — Verify Output Files


In [13]:
import json

# Verify CSV
df_check = pd.read_csv(PROCESSED_DIR / 'qualified_transactions.csv')
print('qualified_transactions.csv:')
print(f'  Rows    : {len(df_check):,}')
print(f'  Columns : {list(df_check.columns)}')
print(f'  CustomerID nulls : {df_check["CustomerID"].isna().sum()}')
print(f'  Qty<=0 rows      : {(df_check["Quantity"] <= 0).sum()}')
print(f'  Price<=0 rows    : {(df_check["UnitPrice"] <= 0).sum()}')
print(f'  Revenue<0 rows   : {(df_check["Revenue"] < 0).sum()}')
print()

# Verify audit JSON
with open(PROCESSED_DIR / 'data_quality_audit.json') as f:
    audit = json.load(f)
print('data_quality_audit.json:')
for k, v in audit.items():
    print(f'  {k}: {v}')

qualified_transactions.csv:
  Rows    : 392,692
  Columns : ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'Revenue', 'IsCancellation']
  CustomerID nulls : 0
  Qty<=0 rows      : 0
  Price<=0 rows    : 0
  Revenue<0 rows   : 0

data_quality_audit.json:
  raw_rows: 541909
  raw_columns: 8
  duplicate_rows: 5268
  missing_customer_id: 135080
  missing_description: 1454
  cancellation_rows: 9288
  non_positive_quantity_rows: 10624
  non_positive_unit_price_rows: 2517
  negative_quantity_rows: 10624
  zero_quantity_rows: 0
  negative_unit_price_rows: 2
  zero_unit_price_rows: 2515
  date_min: 2010-12-01
  date_max: 2011-12-09
  unique_invoices: 25900
  unique_products: 4070
  unique_customers: 4372
  country_count: 38
  qualified_rows: 392692
  qualified_customers: 4338


## Summary

| Stage | Rows |
|---|---|
| Raw loaded | 541,909 |
| After deduplication | 536,641 |
| After qualifying-purchase filter | (see output above) |

**Cleaning actions applied:**
1. InvoiceDate parsed as datetime — 0 failures.
2. Missing Description filled with 'Unknown' — 1,454 rows affected.
3. 5,268 exact duplicate rows removed.
4. IsCancellation flag created for InvoiceNo prefix 'C'.
5. Qualifying-purchase filter applied (CustomerID present, non-cancellation, Quantity>0, UnitPrice>0).
6. Revenue = Quantity × UnitPrice computed — all values ≥ 0.
7. Outputs written to `data/processed/`.
